In [ ]:
"""Animate each selected test/train/validation split as depth-panel MP4s."""

import glob
import json
import os
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

import config
from utils.data import load_dataset
from utils.gridding import df_to_gridded_da
from utils.plotting import animate_depth_panels

In [ ]:
# Load data
df = load_dataset()

# Load the fixed held-out test indices (shared across all folds)
test_split_path = os.path.join(config.output_dir_splits, "test_train_split.json")
with open(test_split_path) as f:
    test_idx = np.array(json.load(f)["test_idx"])

# Assign a colour to each split category
CATEGORY_COLORS = {
    "train": "tab:gray",
    "validation": "tab:blue",
    "test": "tab:red",
    "unobserved": "white",
}
categories = list(CATEGORY_COLORS.keys())
cmap = ListedColormap([CATEGORY_COLORS[c] for c in categories])

In [ ]:
# Iterate over splits
for fname in sorted(glob.glob(os.path.join(config.output_dir_splits, "selected_splits", "fold_*.json"))):
    print(fname)
    with open(fname) as f:
        fold = json.load(f)

    # Tag every grid point with its split role
    temp = df.copy()
    temp["SPLIT"] = "unassigned"
    temp.loc[np.array(fold["train_idx"]), "SPLIT"] = "train"
    temp.loc[np.array(fold["val_idx"]), "SPLIT"] = "validation"
    temp.loc[test_idx, "SPLIT"] = "test"
    temp.loc[temp.P_TEMPERATURE.isna(), "SPLIT"] = "unobserved"
    print(temp["SPLIT"].value_counts())

    # Convert category labels to integer codes so they can be stored in a gridded DataArray
    temp["CAT"] = pd.Categorical(temp["SPLIT"], categories=categories, ordered=True).codes
    da = df_to_gridded_da(df=temp, value_col="CAT")
    animate_depth_panels(da, depth_dim="depth", cmap=cmap, save_as=fname.replace(".json", ".mp4"))
    print()
